In [1]:
pip install python-docx fpdf


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 6.5 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=ed4d5c29218df8627833fa59f0516ead3919248adc98aef0b71ec16dbf10fe45
  Stored in directory: /root/.cache/pip/wheels/65/4f/66/bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf


In [14]:
from docx import Document
from fpdf import FPDF
import re

# Caminho do arquivo Word (.docx)
docx_path = "/content/CONTRATO PD 2022 .docx"

def extrair_texto_sem_tabelas(caminho_arquivo):
    doc = Document(caminho_arquivo)
    texto_extraido = ""
    for paragrafo in doc.paragraphs:
        texto = paragrafo.text.strip()
        if texto and not texto.isspace():
            texto_extraido += texto + "\n\n"
    return texto_extraido

def limpar_campos_preenchimento(texto):
    # Remove sequências de sublinhado (3 ou mais)
    texto = re.sub(r'_{3,}', '', texto)

    # Remove padrões tipo ___/___/___ (datas com sublinhado)
    texto = re.sub(r'_{1,3}/_{1,3}/_{1,4}', '', texto)

    # Remove todos os asteriscos '*' do texto
    texto = texto.replace('*', '')

    # Remove linhas vazias ou só com espaços
    linhas_final = [linha for linha in texto.split('\n') if linha.strip() != '']
    return '\n'.join(linhas_final)

def tratar_caracteres_para_fpdf(texto):
    substituicoes = {
        "–": "-",   # travessão
        "—": "-",
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "…": "...",
        "•": "-",    # transformar bullets em hífen
        "●": "-",    # idem
        "✓": "v",
        "✔": "v",
        "→": "->",
        "←": "<-",
        "°": "º",
        "¼": "1/4",
        "½": "1/2",
        "¾": "3/4",
    }
    for caractere, substituto in substituicoes.items():
        texto = texto.replace(caractere, substituto)
    return texto

class PDFEstiloSimples(FPDF):
    def header(self):
        self.set_font("Arial", "B", 14)
        self.cell(0, 10, "Contrato de Prestação de Serviços", ln=True, align="C")

def main():
    conteudo = extrair_texto_sem_tabelas(docx_path)
    conteudo_limpo = limpar_campos_preenchimento(conteudo)
    conteudo_tratado = tratar_caracteres_para_fpdf(conteudo_limpo)

    pdf = PDFEstiloSimples()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_font("Times", "", 12)
    pdf.multi_cell(0, 8, conteudo_tratado)

    saida_pdf = "Contrato_Tratado.pdf"
    pdf.output(saida_pdf)

    print(f"PDF gerado com sucesso: {saida_pdf}")

if __name__ == "__main__":
    main()

PDF gerado com sucesso: Contrato_Tratado.pdf
